In [11]:
# Student Name: Aaron Baggot
# Student Number: C22716399
# TU856/4

In [12]:
from pathlib import Path
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt


In [13]:
# Save image
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Saving to:", OUT_DIR.resolve())

def show(title, img):
    plt.figure(figsize=(8,5))
    if img.ndim == 3:
        plt.imshow(cv.cvtColoe(img, cv.COLOR_BGR2RGB))
    else:
        plt.imshow(img, cmap="gray")
    plt.title(title);
    plt.axis("off");
    plt.show()

Saving to: /Users/aaronbaggot/Desktop/Image_Processing_Assignment_1/notebooks/outputs


In [14]:
# Load Image
img_path = Path("/Users/aaronbaggot/Desktop/Image_Processing_Assignment_1/data/raw/Shark 1.PNG")  
# img_path = Path("/Users/aaronbaggot/Desktop/Image_Processing_Assignment_1/data/raw/Shark 2.PNG")

I = cv.imread(str(img_path), cv.IMREAD_COLOR)

if I is None:
    raise Exception(f"Image not found: {img_path}")
h, w, = I.shape[:2]

# Resize x 2
S = cv.resize(I, dsize=(2*w, 2*h), interpolation=cv.INTER_CUBIC)

# Crop top left Tested on different values
# C = I[0:min(150, h), 0:min(200, w)]

# C = I[0:min(180, h), 0:min(270, w)]

C = I[0:min(100, h), 0:min(150, w)]


In [15]:
# Rotate 45 degrees on the same canvas
cx, cy = w // 2, h // 2

M = cv.getRotationMatrix2D((cx, cy), 45, 1.0)

R = cv.warpAffine(I, M, (w, h), flags=cv.INTER_LINEAR, borderMode =cv.BORDER_REFLECT)

In [16]:
# Display the outputs
plt.show("Original", I)

In [17]:
plt.show("Upscaled", S)

In [18]:
plt.show("Top left Cropped", C)

In [19]:
plt.show("Rotated", R)

In [20]:
# Save each image with discription
cv.imwrite(str(OUT_DIR / "01_Original.png"), I)
cv.imwrite(str(OUT_DIR / "02_Upscaled.png"), S)
cv.imwrite(str(OUT_DIR / "03_Cropped.png"), C)
cv.imwrite(str(OUT_DIR / "04_Rotated.png"), R)
print("Saved Outputs.")


Saved Outputs.


In [ ]:
# Denoise whcih keeps edges, contrast via CLAHE
den = cv.bilateralFilter(I, d=7, sigmaColor=75, sigmaSpace=75)

LAB = cv.cvtColor(den, cv.COLOR_BGR2LAB)
L, A, B = cv.split(LAB)
L2 = cv.createCLAHE(clipLimit=3.0, tileGridSize=(8,8)).apply(L)
enh = cv.cvtCOLOR(cv.merge([L2, A, B]), cv.COLOR_LAB2BGR)

blur = cv.GaussianBlur(enh, (0,0), 1,0)
enh = cv.addWeighted(enh, 1.4, blur, -0.4, 0)

plt.show("Enhanced", enh)
cv.imwrite(str(OUT_DIR / "05_enhanced.png"), enh)




In [ ]:
# Segment, Otsu on S & LAB-b, morphology, largest component
hsv = cv.cvtColor(enh, cv.COLOR_BGR2HSV); H,S,V = cv.split(hsv)
lab = cv.cvtColor(enh, cv.COLOR_BGR2LAB); L, a, b = cv.split(lab)

_, th_s = cv.threshold(S, 0, 255, cv.THRES_BINARY + cv.THRES_OTSU)
_, th_b = cv.threshold(b, 0, 255, cv.THRES_BINARY + cv.THRES_OTSU)

mask = cv.bitwise_and(th_s, th_b)

k = cv.getStructuringElement(cv.MORPH_ELLIPSE, (7,7))
mask = cv.morphologyEx(mask, cv.MORPH_CLOSE, k, iteration=2
mask = cv.morphologyEx(mask, cv.MORPH_CLOSE, k, iteration=1

num, labels = cv.connectedComponents(mask)
if num > 1:
    sizes = []